# ContractRisk — verificación previa al despliegue

Comprueba el paquete de inferencia con modelos reales en una sesión nueva de **Colab GPU T4**. Integra M1, consulta con fuentes, herramienta y visión FP16. No llama al juez, no usa Gemini ni consume el presupuesto de API. No cambia main ni el Space.

Esta prueba verifica el candidato en T4; no demuestra compatibilidad con el ciclo de vida de ZeroGPU ni reemplaza la evaluación de calidad anterior. Se conservan todas las respuestas, incluidos errores. La captura de prueba se genera a partir de una ficha pública y se identifica como sintética, no como documento firmado.

Ejecutar en orden. Los artefactos de evidencia son el ZIP final y este notebook con sus salidas. La última sección conserva resultados parciales si una celda anterior falla.

In [ ]:
import os,sys,subprocess,tempfile,json
from pathlib import Path
os.environ['USE_TF']='0'
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['ANONYMIZED_TELEMETRY']='false'
ROOT=Path(tempfile.mkdtemp(prefix='contractrisk_deployment16_',dir='/content'))/'repo'
subprocess.run(['git','clone','--depth','1','--branch','Entrega/contractrisk-final','https://github.com/vcastrop/Juan-Camilo-Ramon-Valentina-Castro.git',str(ROOT)],check=True)
os.chdir(ROOT)
COMMIT=subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip()
print('Código:',COMMIT)
def run_visible(arguments,logfile):
    with open(logfile,'w',encoding='utf-8') as log:
        process=subprocess.Popen(arguments,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True);log.write(line);log.flush()
        status=process.wait()
    if status:raise RuntimeError('Proceso detenido. Evidencia conservada en '+logfile)


In [ ]:
run_visible([sys.executable,'-m','pip','install','-r','final/space_candidate/requirements.txt','gradio==6.32.0'],'deployment16_install.log')
run_visible(['apt-get','update'],'deployment16_apt.log')
run_visible(['apt-get','install','-y','fonts-dejavu-core'],'deployment16_fonts.log')
run_visible([sys.executable,'-c',"import torch; assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0), 'Selecciona GPU T4'; print(torch.cuda.get_device_name(0))"],'deployment16_gpu.log')
run_visible([sys.executable,'-m','unittest','discover','-s','final/tests','-v'],'deployment16_tests_final.log')
run_visible([sys.executable,'-m','unittest','discover','-s','m3/tests','-v'],'deployment16_tests_m3.log')


In [ ]:
CANDIDATE=ROOT/'deployment16_candidate'
run_visible([sys.executable,'final/build_space_candidate.py','--output',str(CANDIDATE)],'deployment16_build.log')
RESULTS=ROOT/'deployment16_results'
run_visible([sys.executable,'-u','final/run_service_preflight.py','--candidate',str(CANDIDATE),'--output',str(RESULTS)],'deployment16_preflight.log')


## Evidencia e interpretación

`completed_requires_review` significa que terminaron las peticiones sin errores técnicos inesperados, no que las respuestas sean correctas. Los controles, fuentes y uso real de la herramienta se revisan en `requests.jsonl` y `metadata.json`. `failed_partial_evidence_retained` conserva lo producido antes del fallo.

El paquete de cuatro archivos no debe publicarse aún en el Space: falta revisar esta ejecución y verificar después el arranque y las peticiones en ZeroGPU. La telemetría de este notebook no se presenta como evidencia de peticiones públicas.

In [ ]:
import zipfile
from google.colab import files
metadata=ROOT/'deployment16_results/metadata.json'
print(metadata.read_text(encoding='utf-8') if metadata.exists() else 'No se creó metadata; consultar los logs conservados.')
archive=ROOT.parent/'contractrisk_final_despliegue16_resultados.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for folder in ['deployment16_candidate','deployment16_results']:
        base=ROOT/folder
        if base.exists():
            for item in sorted(base.rglob('*')):
                if item.is_file():z.write(item,item.relative_to(ROOT))
    for item in sorted(ROOT.glob('deployment16_*.log')):z.write(item,item.name)
    z.writestr('checkout_commit.txt',COMMIT+'\n')
files.download(str(archive))
